# 实验七：Jetson 机械臂指令接收端

此 notebook **只在 Jetson 的 Jupyter 内运行**，不调用摄像头。电脑本地的识别 notebook 通过 USB 虚拟网卡向 `192.168.55.1:8765` 发送指令。先在演练模式联调，确认后手动点击“连接并启用真机”。

映射：One→张开夹爪，Two→闭合夹爪，Three→回到零点，OK→确认，Five→停止锁定。停止锁定会拒绝新动作，已开始的 SDK 动作无法由 HTTP 中断；真实急停请使用机械臂电源/急停装置。

In [ ]:
from pathlib import Path
import csv
import json
import secrets
import sys
import threading
from datetime import datetime
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
import ipywidgets as widgets
from IPython.display import display

目录 = Path('/home/jetson/Notebook/5.目标识别')
机械臂目录 = Path('/home/jetson/Notebook/4.机械臂OpenCV视觉处理实验')
记录路径 = 目录 / 'week7_remote_command_log.csv'
监听地址, 监听端口 = '0.0.0.0', 8765
动作表 = {'open_gripper': '张开夹爪', 'close_gripper': '闭合夹爪', 'home': '回到零点'}
令牌 = secrets.token_urlsafe(24)
状态锁 = threading.RLock()
状态 = {'模式': '演练', '忙碌': False, '停止锁定': False, '最后动作': '无', '最后结果': '尚未执行'}
机械臂 = None
服务 = None
服务线程 = None

def 写日志(事件, 动作='', 结果=''):
    """只记录实际收到的请求与执行结果；演练和真机明确区分。"""
    新文件 = not 记录路径.exists()
    with 记录路径.open('a', newline='', encoding='utf-8-sig') as 文件:
        表格 = csv.writer(文件)
        if 新文件:
            表格.writerow(['时间', '事件', '动作', '结果', '模式'])
        表格.writerow([datetime.now().isoformat(timespec='seconds'), 事件, 动作, 结果, 状态['模式']])

def 执行动作(指令, 执行模式):
    """在独立线程中执行固定动作，避免占住 HTTP 服务。"""
    global 机械臂
    动作 = 动作表[指令]
    try:
        if 执行模式 == '真机':
            {'open_gripper': 机械臂.gripper_open,
             'close_gripper': 机械臂.gripper_close,
             'home': 机械臂.home}[指令]()
        结果 = '演练已接收' if 执行模式 == '演练' else '真机动作已返回'
    except Exception as 错误:
        结果 = '执行失败：' + str(错误)
    with 状态锁:
        状态['最后结果'] = 结果
        状态['忙碌'] = False
        写日志('执行结束', 动作, 结果)

class 指令处理器(BaseHTTPRequestHandler):
    def log_message(self, 格式, *参数):
        pass

    def 回复(self, 状态码, 内容):
        数据 = json.dumps(内容, ensure_ascii=False).encode('utf-8')
        self.send_response(状态码)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(数据)))
        self.end_headers()
        self.wfile.write(数据)

    def 已授权(self):
        return secrets.compare_digest(self.headers.get('X-Week7-Token', ''), 令牌)

    def do_GET(self):
        if not self.已授权():
            self.回复(401, {'错误': '令牌不正确'})
            return
        if self.path != '/status':
            self.回复(404, {'错误': '路径不存在'})
            return
        with 状态锁:
            self.回复(200, dict(状态))

    def do_POST(self):
        if not self.已授权():
            self.回复(401, {'错误': '令牌不正确'})
            return
        if self.path != '/command':
            self.回复(404, {'错误': '路径不存在'})
            return
        try:
            长度 = int(self.headers.get('Content-Length', '0'))
            if not 0 < 长度 <= 1024:
                raise ValueError('请求大小不正确')
            指令 = json.loads(self.rfile.read(长度).decode('utf-8'))['command']
        except (ValueError, KeyError, UnicodeDecodeError, TypeError):
            self.回复(400, {'错误': '请求格式不正确'})
            return
        with 状态锁:
            if 指令 == 'stop':
                状态['停止锁定'] = True
                写日志('停止锁定', '', '拒绝后续动作；执行中的动作不会中断')
                self.回复(200, dict(状态))
                return
            if 指令 == 'clear_stop':
                if 状态['忙碌']:
                    self.回复(409, {'错误': '动作执行中，不能解除停止'})
                    return
                状态['停止锁定'] = False
                写日志('解除停止')
                self.回复(200, dict(状态))
                return
            if 指令 not in 动作表:
                self.回复(400, {'错误': '不支持的指令'})
                return
            if 状态['停止锁定'] or 状态['忙碌']:
                self.回复(409, {'错误': '停止锁定或动作执行中'})
                return
            状态['忙碌'] = True
            状态['最后动作'] = 动作表[指令]
            状态['最后结果'] = '执行中'
            模式 = 状态['模式']
            写日志('接受指令', 动作表[指令], '执行中')
            threading.Thread(target=执行动作, args=(指令, 模式), daemon=True).start()
            self.回复(202, {'结果': '已接受', '模式': 模式, '动作': 动作表[指令]})

def 启动服务(_=None):
    global 服务, 服务线程
    if 服务线程 is not None and 服务线程.is_alive():
        提示.value = '服务已经运行。'
        return
    try:
        服务 = ThreadingHTTPServer((监听地址, 监听端口), 指令处理器)
        服务.daemon_threads = True
        服务线程 = threading.Thread(target=服务.serve_forever, daemon=True)
        服务线程.start()
        提示.value = '接收端已启动；当前模式：' + 状态['模式']
    except OSError as 错误:
        提示.value = '启动失败：' + str(错误)

def 停止服务(_=None):
    global 服务, 服务线程
    with 状态锁:
        状态['停止锁定'] = True
    if 服务 is not None:
        服务.shutdown()
        服务.server_close()
        服务 = None
        服务线程 = None
    提示.value = '服务已停止；已发送的机械臂动作不能由此中断。'

def 启用真机(_):
    global 机械臂
    with 状态锁:
        if 状态['忙碌']:
            提示.value = '动作正在执行，暂不能切换。'
            return
    try:
        if 机械臂 is None:
            sys.path.insert(0, str(机械臂目录))
            from arm5dof_uservo import Arm5DoFUServo
            机械臂 = Arm5DoFUServo(is_init_pose=False, config_folder=str(机械臂目录 / 'config'))
        with 状态锁:
            状态['模式'] = '真机'
        提示.value = '已连接机械臂；连接时未发送运动指令。'
    except (Exception, SystemExit) as 错误:
        提示.value = '机械臂连接失败：' + str(错误)

def 切换演练(_):
    with 状态锁:
        if 状态['忙碌']:
            提示.value = '动作正在执行，暂不能切换。'
            return
        状态['模式'] = '演练'
    提示.value = '已切换到演练模式。'

启动按钮 = widgets.Button(description='启动接收服务')
停止按钮 = widgets.Button(description='停止接收服务')
真机按钮 = widgets.Button(description='连接并启用真机')
演练按钮 = widgets.Button(description='演练模式')
提示 = widgets.HTML(value='等待启动；默认演练模式。')
启动按钮.on_click(启动服务)
停止按钮.on_click(停止服务)
真机按钮.on_click(启用真机)
演练按钮.on_click(切换演练)
display(widgets.VBox([widgets.HBox([启动按钮, 停止按钮, 真机按钮, 演练按钮]),
    widgets.HTML(value='<b>USB 接收地址：</b>http://192.168.55.1:8765'),
    widgets.HTML(value='<b>本次连接令牌：</b>' + 令牌), 提示]))

保持本 notebook 的 Jetson 内核运行。把上方连接令牌复制到电脑本地 notebook；每次重新运行代码单元，令牌会变化。先完成演练联调，再将机械臂周围清空并手动启用真机。接收日志保存在 `week7_remote_command_log.csv`，模式列会区分演练与真机。